# Out-of-Distribution Evaluation
## Privify v0.1 fine-tuned face detector vs COCO person baseline

Measures the **recall** of two detectors on the deployment-target video
(`samples/input.mp4`, a CCTV street-level scene) to **quantify the domain
shift** identified qualitatively during pipeline testing.

The two detectors evaluated against the same ground truth are:

1. **`face-detector-v0.1`** — our YOLOv8n fine-tuned on WIDER FACE
   (single class `face`).
2. **`yolov8n.pt` COCO** — pre-trained baseline, filtered to class
   `person` (`classes=[0]`).

**Ground truth.** For each sampled frame a human annotator counts the
*whole identifiable persons* present — i.e. every subject that, in a real
CCTV scene, should be anonymized for GDPR compliance (partial bodies
included when clearly a person). This count is the denominator of the recall.

**Why person-count, not face-count.** The fine-tuned detector finds *faces*;
the COCO baseline finds *persons*. Anchoring both to the same
GDPR-motivated person count makes the metric measure what actually matters
for anonymization coverage — and exposes the failure mode of a frontal-face
detector on a surveillance scene where many subjects show no usable face.

The notebook runs **both locally** (repo as working directory) **and on
Google Colab** (clone + install cell below). Outputs (comparison table and
figures) are written to `evaluation/` for reuse in the Technical Analysis
Document (sections 3 *Experimental Results* and 4 *Failure Analysis*).

In [ ]:
# --- Environment setup: works both locally and on Google Colab ------------
import os
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_DIR = "/content/privify"
    REPO_URL = "https://github.com/jenz26/privify.git"
    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}
    # ultralytics pulls in torch, opencv, matplotlib and numpy.
    !pip install -q ultralytics
    PROJECT_ROOT = Path(REPO_DIR)
else:
    # Local: resolve the repo root whether the cwd is the root or notebooks/.
    PROJECT_ROOT = Path.cwd()
    if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
        PROJECT_ROOT = PROJECT_ROOT.parent
    os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# --- Imports (standard project stack only — no pandas) --------------------
import json

import cv2
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

from src.detector import Detector

# --- Configuration --------------------------------------------------------
VIDEO_PATH = Path("samples/input.mp4")
N_FRAMES = 10  # number of frames sampled from the video
CONF_THRESHOLD = 0.25  # detection confidence threshold (both models)

OUTPUT_DIR = Path("evaluation")
FRAMES_DIR = OUTPUT_DIR / "frames"  # extracted PNGs (gitignored)
FIGURES_DIR = OUTPUT_DIR / "figures"  # TAD figures (versioned)
GT_PATH = OUTPUT_DIR / "ground_truth.json"  # manual annotations (versioned)

for _d in (FRAMES_DIR, FIGURES_DIR):
    _d.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Video        : {VIDEO_PATH}  (exists: {VIDEO_PATH.exists()})")
print(f"Running on   : {'Colab' if IN_COLAB else 'local'}")


# --- Small helpers reused across cells ------------------------------------
def load_frame_bgr(sample: int) -> np.ndarray:
    """Load a previously extracted frame (BGR) from disk by its sample ordinal."""
    path = FRAMES_DIR / f"frame_{sample:02d}.png"
    img = cv2.imread(str(path))
    if img is None:
        raise FileNotFoundError(f"Frame not found: {path}. Run the frame-extraction cell first.")
    return img


def draw_boxes(
    frame_bgr: np.ndarray,
    boxes: list[tuple[int, int, int, int]],
    color_bgr: tuple[int, int, int],
) -> np.ndarray:
    """Return an RGB copy of *frame_bgr* with *boxes* drawn in *color_bgr*."""
    img = frame_bgr.copy()
    for x1, y1, x2, y2 in boxes:
        cv2.rectangle(img, (x1, y1), (x2, y2), color_bgr, 2)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


def to_markdown_table(rows: list[dict], float_fmt: str = "{:.3f}") -> str:
    """Build a GitHub-flavored Markdown table from uniform dict rows.

    A lightweight, dependency-free substitute for a printed DataFrame: it keeps
    the notebook within the project stack (no pandas) while still rendering as a
    real table via ``IPython.display.Markdown``.
    """
    if not rows:
        return "_(no rows)_"
    columns = list(rows[0].keys())

    def cell(value: object) -> str:
        if isinstance(value, float):
            return "nan" if value != value else float_fmt.format(value)
        return str(value)

    header = "| " + " | ".join(columns) + " |"
    separator = "| " + " | ".join("---" for _ in columns) + " |"
    body = ["| " + " | ".join(cell(row[c]) for c in columns) + " |" for row in rows]
    return "\n".join([header, separator, *body])

## Step 1 — Frame extraction

Sample `N_FRAMES` frames at **uniform, evenly-spaced** positions across the
clip. Absolute indices are computed as `(i + 0.5) · total / N` for
`i = 0 … N-1`, which spreads the samples across the whole timeline while
skipping the very first and last frames (often blurred intro/outro).

The frame count reported by OpenCV (`CAP_PROP_FRAME_COUNT`) is unreliable for
some codecs, so we fall back to a full-decode count when it is missing, and
read each target frame with a sequential-decode fallback if the seek fails.

Frames are saved **by sample ordinal** as
`evaluation/frames/frame_{i:02d}.png` (`i = 0 … N-1`). Keying by ordinal —
not by the absolute frame number — keeps the ground truth **portable**: if a
different OpenCV/codec build reports a slightly different frame count, the
sample ordinals still line up, so an annotation made in one environment stays
valid in another (the next cell only *warns* on such drift).

In [ ]:
cap = cv2.VideoCapture(str(VIDEO_PATH))
if not cap.isOpened():
    raise FileNotFoundError(f"Could not open video: {VIDEO_PATH.resolve()}")

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)

# CAP_PROP_FRAME_COUNT can be 0 or a bad estimate for some codecs: count by
# decoding the whole clip once, then rewind.
if total_frames <= 0:
    total_frames = 0
    while cap.read()[0]:
        total_frames += 1
    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
if total_frames <= 0:
    cap.release()
    raise RuntimeError(f"No decodable frames in {VIDEO_PATH}")
print(f"Video has {total_frames} frames at {fps:.1f} fps")

# Evenly-spaced, deterministic sampling. For total=300, N=10 -> 15,45,...,285.
frame_indices = [int((i + 0.5) * total_frames / N_FRAMES) for i in range(N_FRAMES)]
print("Sampled absolute frame indices:", frame_indices)


def _read_frame_at(capture: cv2.VideoCapture, target: int) -> np.ndarray:
    """Return the frame at *target*, falling back to sequential decode."""
    capture.set(cv2.CAP_PROP_POS_FRAMES, target)
    ok, frame = capture.read()
    if ok:
        return frame
    # Seek unsupported for this codec: decode sequentially up to *target*.
    capture.set(cv2.CAP_PROP_POS_FRAMES, 0)
    frame = None
    for _ in range(target + 1):
        ok, current = capture.read()
        if not ok:
            break
        frame = current
    if frame is None:
        raise RuntimeError(f"Could not read frame {target} from {VIDEO_PATH}")
    return frame


frames: dict[int, np.ndarray] = {}
for i, target in enumerate(frame_indices):
    frame = _read_frame_at(cap, target)
    frames[i] = frame
    cv2.imwrite(str(FRAMES_DIR / f"frame_{i:02d}.png"), frame)
cap.release()
print(
    f"Saved {len(frames)} frames to {FRAMES_DIR} " f"(frame_00.png .. frame_{N_FRAMES - 1:02d}.png)"
)

# Preview grid (5 columns x 2 rows), keyed by sample ordinal.
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
for ax, i in zip(axes.flat, range(N_FRAMES), strict=False):
    ax.imshow(cv2.cvtColor(frames[i], cv2.COLOR_BGR2RGB))
    ax.set_title(f"sample {i} (frame {frame_indices[i]})")
    ax.axis("off")
fig.suptitle("Sampled frames (uniform spacing)", fontsize=14)
plt.tight_layout()
plt.show()

## Step 2 — Manual ground-truth annotation

For **each** of the extracted frames (`evaluation/frames/frame_00.png` …
`frame_09.png`), open the PNG and count the number of **whole, identifiable
persons** visible. The counting rule:

- Count every subject clearly recognizable as a person, **including partial
  bodies** (e.g. someone half-occluded or partly out of frame) when there is
  no doubt it is a person.
- The mental model is: *"how many subjects in this CCTV scene should be
  anonymized to be GDPR-compliant?"* — that number is the ground truth.
- This is **not** a face count: a person facing away, far from the camera, or
  with an occluded face still counts as one subject to anonymize.

Write each count into `evaluation/ground_truth.json` (created by the next
cell as a placeholder), filling `person_count` for every `sample`. Each entry
also records the absolute `frame_idx` for reference. This file is
**versioned** — it is the human reference both detectors are scored against.

In [ ]:
if not GT_PATH.exists():
    placeholder = {
        "video": VIDEO_PATH.as_posix(),
        "n_frames": N_FRAMES,
        "sampling": "uniform-evenly-spaced",
        "criterion": (
            "whole identifiable persons (partial bodies included) — " "GDPR anonymization target"
        ),
        "annotations": [
            {"sample": i, "frame_idx": frame_indices[i], "person_count": None, "notes": ""}
            for i in range(N_FRAMES)
        ],
    }
    GT_PATH.write_text(json.dumps(placeholder, indent=2), encoding="utf-8")
    print(f"Created placeholder: {GT_PATH}")
    print(
        "ACTION REQUIRED -> open it, fill 'person_count' for every sample, "
        "then re-run this cell."
    )

gt_doc = json.loads(GT_PATH.read_text(encoding="utf-8"))
annotations = gt_doc["annotations"]

# Annotations are keyed by sample ordinal (0..N-1), NOT by the absolute frame
# index, so a ground truth annotated in one environment stays valid in another
# even if the codec reports a slightly different frame count.
samples = {a["sample"] for a in annotations}
if samples != set(range(N_FRAMES)):
    raise ValueError(
        f"ground_truth.json covers samples {sorted(samples)} but N_FRAMES={N_FRAMES}. "
        "Delete evaluation/ground_truth.json and re-run from Step 1."
    )

# Soft check: warn (do not fail) if absolute indices drifted vs the annotation
# environment (different OpenCV/codec frame count). Counts stay valid because
# the drift is at most a frame or two — adjacent frames show the same subjects.
drift = [
    (a["sample"], a["frame_idx"], frame_indices[a["sample"]])
    for a in annotations
    if a.get("frame_idx") is not None and a["frame_idx"] != frame_indices[a["sample"]]
]
if drift:
    print("WARNING: absolute frame indices drifted vs the annotation environment:")
    for s, was, now in drift:
        print(f"  sample {s}: annotated frame {was} -> extracted frame {now}")
    print("Counts remain valid (adjacent frames); proceeding.")

missing = [a["sample"] for a in annotations if a["person_count"] is None]
if missing:
    raise ValueError(
        f"person_count is still empty for samples {missing}. "
        f"Edit {GT_PATH} (manual annotation), then re-run this cell."
    )

ground_truth = {a["sample"]: int(a["person_count"]) for a in annotations}
frame_abs = {a["sample"]: a.get("frame_idx") for a in annotations}
print("Ground truth (sample -> person_count):", ground_truth)
print("Total persons across sampled frames:", sum(ground_truth.values()))

## Step 3 — Inference with `face-detector-v0.1` (fine-tuned)

Run our fine-tuned single-class face detector on every sampled frame and
record the number of `face` detections per frame. The `Detector` wrapper
downloads the fine-tuned weights from the GitHub Release on first use and
caches them in `models/`.

In [ ]:
detector_face = Detector(conf_threshold=CONF_THRESHOLD)

face_boxes: dict[int, list[tuple[int, int, int, int]]] = {}
for i in sorted(ground_truth):
    dets = detector_face.detect(load_frame_bgr(i))
    face_boxes[i] = [d.bbox for d in dets]
    print(
        f"sample {i} (frame {frame_abs[i]}): {len(dets)} face detection(s) "
        f"(GT persons: {ground_truth[i]})"
    )

# Three example frames (first / middle / last sampled) reused for comparison.
samples_sorted = sorted(ground_truth)
EXAMPLE_SAMPLES = [samples_sorted[0], samples_sorted[len(samples_sorted) // 2], samples_sorted[-1]]

FACE_COLOR_BGR = (0, 200, 0)  # green
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, i in zip(axes, EXAMPLE_SAMPLES, strict=True):
    ax.imshow(draw_boxes(load_frame_bgr(i), face_boxes[i], FACE_COLOR_BGR))
    ax.set_title(
        f"face-detector-v0.1 — frame {frame_abs[i]}\n"
        f"{len(face_boxes[i])} det / GT {ground_truth[i]}"
    )
    ax.axis("off")
fig.suptitle("Fine-tuned face detector", fontsize=14)
plt.tight_layout()
plt.show()

## Step 4 — Inference with the COCO `person` baseline

Load the stock `yolov8n.pt` (COCO-pretrained) **directly** — bypassing the
`Detector` wrapper, which by design fetches our fine-tuned face weights.
Ultralytics downloads `yolov8n.pt` automatically on first use and caches it
in the user directory (it is **not** added to the repository). Predictions
are filtered to COCO class `0` (`person`) via `classes=[0]`.

In [ ]:
from ultralytics import YOLO

coco_model = YOLO("yolov8n.pt")  # auto-downloaded & cached by ultralytics

person_boxes: dict[int, list[tuple[int, int, int, int]]] = {}
for i in sorted(ground_truth):
    results = coco_model.predict(load_frame_bgr(i), conf=CONF_THRESHOLD, classes=[0], verbose=False)
    boxes = results[0].boxes
    person_boxes[i] = [tuple(int(v) for v in xyxy) for xyxy in boxes.xyxy.tolist()]
    print(
        f"sample {i} (frame {frame_abs[i]}): {len(person_boxes[i])} person detection(s) "
        f"(GT persons: {ground_truth[i]})"
    )

# Same example frames as Step 3, for side-by-side visual comparison.
PERSON_COLOR_BGR = (255, 0, 0)  # blue (BGR)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, i in zip(axes, EXAMPLE_SAMPLES, strict=True):
    ax.imshow(draw_boxes(load_frame_bgr(i), person_boxes[i], PERSON_COLOR_BGR))
    ax.set_title(
        f"COCO person — frame {frame_abs[i]}\n" f"{len(person_boxes[i])} det / GT {ground_truth[i]}"
    )
    ax.axis("off")
fig.suptitle("COCO person baseline", fontsize=14)
plt.tight_layout()
plt.show()

## Step 5 — Recall computation

We only have **counts** as ground truth (not matched boxes), so we use a
**count-based recall**: the fraction of GT subjects covered by detections,

> recall = min(detections, ground_truth) / ground_truth

The `min(·)` clip keeps recall in `[0, 1]`: a detector cannot "recall" more
subjects than exist, and surplus detections (false positives, or several
faces inferred for fewer people) must not inflate the score above 1. The raw
detection counts are kept in the table so nothing is hidden.

In [ ]:
def count_recall(n_detections: int, n_truth: int) -> float:
    """Count-based recall clipped to [0, 1]; NaN when there is no subject."""
    if n_truth == 0:
        return float("nan")
    return min(n_detections, n_truth) / n_truth


rows = []
for i in sorted(ground_truth):
    gt = ground_truth[i]
    n_face = len(face_boxes[i])
    n_person = len(person_boxes[i])
    rows.append(
        {
            "sample": i,
            "frame_idx": frame_abs[i],
            "ground_truth": gt,
            "face_detections": n_face,
            "face_recall": count_recall(n_face, gt),
            "person_detections": n_person,
            "person_recall": count_recall(n_person, gt),
        }
    )

face_recalls = np.array([r["face_recall"] for r in rows], dtype=float)
person_recalls = np.array([r["person_recall"] for r in rows], dtype=float)

agg_rows = [
    {
        "stat": name,
        "face_recall": float(fn(face_recalls)),
        "person_recall": float(fn(person_recalls)),
    }
    for name, fn in (
        ("mean", np.nanmean),
        ("std", lambda a: np.nanstd(a, ddof=1)),
        ("min", np.nanmin),
        ("max", np.nanmax),
    )
]

display(Markdown("**Per-frame results**\n\n" + to_markdown_table(rows)))
display(Markdown("**Aggregate recall**\n\n" + to_markdown_table(agg_rows)))

## Step 6 — Comparison visualization

A grouped bar chart of per-frame recall (fine-tuned face vs COCO person)
plus side-by-side detection overlays on the three example frames. Both
figures are saved to `evaluation/figures/` for direct inclusion in the TAD.

In [ ]:
# --- Bar chart: per-frame recall ------------------------------------------
face_recall_vals = [r["face_recall"] for r in rows]
person_recall_vals = [r["person_recall"] for r in rows]
frame_labels = [r["frame_idx"] for r in rows]
x = np.arange(len(rows))
width = 0.38

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - width / 2, face_recall_vals, width, label="face-detector-v0.1", color="tab:green")
ax.bar(x + width / 2, person_recall_vals, width, label="COCO person baseline", color="tab:blue")
ax.axhline(1.0, ls="--", lw=1, color="grey")
ax.set_xticks(x)
ax.set_xticklabels(frame_labels)
ax.set_xlabel("Frame index")
ax.set_ylabel("Recall (count-based, clipped to [0, 1])")
ax.set_ylim(0, 1.1)
ax.set_title("Per-frame recall on the CCTV deployment target")
ax.legend()
plt.tight_layout()
fig.savefig(FIGURES_DIR / "recall_per_frame.png", dpi=150, bbox_inches="tight")
plt.show()

# --- Side-by-side detection overlays on the example frames ----------------
n_ex = len(EXAMPLE_SAMPLES)
fig, axes = plt.subplots(n_ex, 2, figsize=(14, 5 * n_ex))
for row, i in enumerate(EXAMPLE_SAMPLES):
    axes[row, 0].imshow(draw_boxes(load_frame_bgr(i), face_boxes[i], FACE_COLOR_BGR))
    axes[row, 0].set_title(
        f"frame {frame_abs[i]} — face-detector-v0.1 "
        f"({len(face_boxes[i])} det / GT {ground_truth[i]})"
    )
    axes[row, 0].axis("off")
    axes[row, 1].imshow(draw_boxes(load_frame_bgr(i), person_boxes[i], PERSON_COLOR_BGR))
    axes[row, 1].set_title(
        f"frame {frame_abs[i]} — COCO person "
        f"({len(person_boxes[i])} det / GT {ground_truth[i]})"
    )
    axes[row, 1].axis("off")
fig.suptitle("Fine-tuned face (left) vs COCO person (right)", fontsize=14)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "side_by_side_examples.png", dpi=150, bbox_inches="tight")
plt.show()

print("Figures saved to", FIGURES_DIR)

## Summary

Final aggregated comparison and the headline numbers used in the TAD.

In [ ]:
face_mean = float(np.nanmean([r["face_recall"] for r in rows]))
person_mean = float(np.nanmean([r["person_recall"] for r in rows]))
gap_pp = (person_mean - face_mean) * 100

print("=" * 64)
print("OUT-OF-DISTRIBUTION RECALL — CCTV deployment target")
print("=" * 64)
print(f"Fine-tuned face detector (face-detector-v0.1) mean recall: {face_mean:.1%}")
print(f"COCO person baseline mean recall:                         {person_mean:.1%}")
print(f"Gap (person - face):                                      {gap_pp:.1f} percentage points")
display(Markdown("**Per-frame results**\n\n" + to_markdown_table(rows)))
print(
    "\nInterpretation: a large positive gap quantifies the domain shift — "
    "the frontal-face detector misses subjects whose faces are not usable "
    "in a surveillance scene, while a generic person detector still covers "
    "them. See TAD sections 3 (Experimental Results) and 4 (Failure Analysis)."
)